# Pembersihan Data

Perkiraan waktu: **45** menit

Data nyata yang dikumpulkan di lapangan jarang rapi. Isinya sering berisi baris ganda, nilai kosong, dan pencatatan yang tidak relevan. Kalau data seperti ini langsung dipakai untuk pelatihan model, akurasi prediksi akan turun. Karena itu pembersihan data dikerjakan sebelum tahap pemodelan.

Notebook ini membahas teknik pembersihan data yang paling sering dipakai.

## Tujuan

Setelah menyelesaikan latihan ini, kamu dapat:

- Mentransformasi data memakai fungsi log
- Menangani data duplikat
- Menangani nilai yang hilang
- Melakukan standardisasi dan normalisasi
- Menangani outlier


## Pustaka yang dipakai

- `pandas` untuk mengelola tabel data.
- `numpy` untuk operasi matematika.
- `seaborn` dan `matplotlib` untuk visualisasi.
- `scikit-learn` untuk penskalaan fitur.
- `scipy` untuk perhitungan statistik.


In [ ]:
# Imports
import urllib.request
import warnings
from pathlib import Path

import matplotlib.pylab as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler

warnings.filterwarnings("ignore")


## Membaca dan memahami data

Data yang dipakai adalah `Ames_Housing_Data1.tsv`, yaitu catatan penjualan rumah di Ames, Iowa, pada rentang 2006 sampai 2010.

Berkas dibaca ke DataFrame pandas, lalu lima baris pertama ditampilkan.


In [ ]:
# Unduh data sekali saja, lalu muat ke DataFrame
data_path = Path("Ames_Housing_Data1.tsv")
data_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML0232EN-SkillsNetwork/asset/Ames_Housing_Data1.tsv"

if not data_path.exists():
    urllib.request.urlretrieve(data_url, data_path)

dataset = pd.read_csv(data_path, sep="\t")
dataset.head()


Ringkasan tipe data dan jumlah nilai terisi bisa dilihat lewat `info()`.


In [ ]:
dataset.info()


Hasil di atas menunjukkan 2930 baris dengan 81 fitur. Kolom `Non-Null Count` menyatakan jumlah nilai yang terisi; jika angkanya 2930, kolom tersebut tidak punya nilai kosong. `SalePrice` adalah variabel target, sedangkan kolom lain berperan sebagai prediktor. Tipe datanya campuran: numerik (`int64` dan `float64`) serta kategorikal (`object`).


Statistik ringkas `SalePrice` diperoleh dengan `describe()`.


In [ ]:
dataset["SalePrice"].describe()


Nilai minimum masih di atas nol, tetapi jarak antara minimum dan persentil ke-25 lebih besar daripada jarak antara persentil ke-75 dan nilai maksimum. Pola ini menandakan distribusi yang tidak simetris, sehingga normalitas `SalePrice` perlu diperiksa pada bagian transformasi log.


## Latihan 1

`describe()` hanya merangkum kolom numerik. Untuk kolom kategorikal, gunakan `value_counts()`. Tampilkan sebaran kategori pada kolom `Sale Condition`.


In [ ]:
dataset["Sale Condition"].value_counts()


## Mencari korelasi

Sebelum membersihkan data, ada baiknya mengukur keeratan hubungan antara `SalePrice` dan setiap prediktor. Fitur yang korelasinya lemah biasanya tidak banyak membantu analisis. Hubungan ini bisa dilihat lewat scatter plot, heat map, matriks korelasi, atau koefisien korelasi Pearson. Perhitungan korelasi hanya berlaku untuk kolom numerik, jadi kolom kategorikal disisihkan lebih dulu.


In [ ]:
# Ambil kolom numerik, lalu urutkan korelasi terhadap SalePrice
numeric_data = dataset.select_dtypes(include=["float64", "int64"])
price_correlation = numeric_data.corr()["SalePrice"].drop("SalePrice")
strong_features = price_correlation[price_correlation.abs() > 0.5].sort_values(ascending=False)

print(f"Jumlah fitur dengan korelasi di atas 0.5: {len(strong_features)}")
print(strong_features)


Ada 11 fitur dengan koefisien korelasi di atas 0.5. Fitur-fitur ini berhubungan erat dengan harga jual.


Hubungan tersebut bisa diperiksa secara visual memakai `pairplot()`. Pair plot juga berguna untuk melihat titik yang menyimpang dari pola umum.


In [ ]:
# Pecah kolom menjadi beberapa kelompok agar plot tetap terbaca
for start in range(0, len(numeric_data.columns), 5):
    sns.pairplot(
        data=numeric_data,
        x_vars=numeric_data.columns[start:start + 5],
        y_vars=["SalePrice"],
    )


Dari koefisien korelasi dan pair plot, fitur yang paling erat terkait `SalePrice` adalah `Overall Qual`, `Gr Liv Area`, `Garage Cars`, dan `Garage Area`.


## Transformasi log

Analisis regresi mengasumsikan data berdistribusi normal. Distribusi `SalePrice` diperiksa secara visual dengan histogram dan kurva kepadatan.


In [ ]:
sns.histplot(dataset["SalePrice"], kde=True)


Kurva di atas belum simetris. Ekornya memanjang ke kanan, jadi distribusinya miring positif. Ukuran kemiringan ini disebut *skewness*. Selain skewness, dikenal juga *kurtosis*, yaitu tingkat keruncingan puncak kurva. Keduanya sering dipakai bersama untuk menggambarkan bentuk distribusi.


In [ ]:
print(f"Skewness: {dataset['SalePrice'].skew():f}")


Distribusi disebut cukup simetris bila skewness berada antara -0.5 dan 0.5, miring sedang antara 0.5 sampai 1.0 (atau -0.5 sampai -1.0), dan sangat miring bila melewati 1.0 (atau di bawah -1.0). Nilai `SalePrice` sekitar 1.7, jadi tergolong sangat miring.

Transformasi log lewat `np.log()` dapat membuat distribusinya lebih mendekati normal.


In [ ]:
log_price = np.log(dataset["SalePrice"])


In [ ]:
sns.histplot(log_price, kde=True)


In [ ]:
print(f"Skewness: {log_price.skew():f}")


Setelah transformasi log, kurvanya jauh lebih simetris dan skewness turun ke sekitar -0.01.

Selain log, kemiringan data juga bisa dikoreksi dengan akar kuadrat (`np.sqrt`) atau transformasi Box-Cox (`stats.boxcox`).


## Latihan 2

Periksa kolom `Lot Area` secara visual. Bila distribusinya miring, terapkan transformasi log lalu bandingkan nilai skewness sebelum dan sesudahnya.


In [ ]:
sns.histplot(dataset["Lot Area"], kde=True)
print(f"Skewness sebelum: {dataset['Lot Area'].skew():f}")

log_lot_area = np.log(dataset["Lot Area"])
print(f"Skewness sesudah: {log_lot_area.skew():f}")


## Menangani duplikat

Baris ganda membuat bobot satu pengamatan menjadi berlebih, sehingga hasil analisis ikut bergeser. Duplikat dideteksi lewat `duplicated()` pada kolom `PID`, yaitu nomor unik setiap properti.


In [ ]:
duplicate_rows = dataset[dataset.duplicated(["PID"])]
duplicate_rows


Ada satu baris ganda. Baris tersebut dibuang dengan `drop_duplicates()`, yang secara bawaan membandingkan seluruh kolom.


In [ ]:
deduplicated = dataset.drop_duplicates()
deduplicated


Duplikasi pada indeks dapat diperiksa dengan atribut `index.is_unique`.


In [ ]:
dataset.index.is_unique


## Latihan 3

Hapus duplikat hanya berdasarkan satu kolom, misalnya kolom `Order`, dengan mengisi parameter `subset`.


In [ ]:
deduplicated_by_order = dataset.drop_duplicates(subset=["Order"])
deduplicated_by_order.shape


## Menangani nilai hilang

pandas menyediakan `isna()`, `isnull()`, dan `notna()` untuk mendeteksi nilai kosong. Untuk melihat kolom mana yang paling banyak kosong, jumlah nilai kosong diurutkan lalu digambarkan sebagai diagram batang.


In [ ]:
# Hitung nilai kosong per kolom, lalu plot 20 kolom teratas
missing_counts = dataset.isnull().sum().sort_values(ascending=False)
missing_counts.head(20).plot(kind="bar", figsize=(8, 6), fontsize=10)

plt.xlabel("Kolom", fontsize=20)
plt.ylabel("Jumlah", fontsize=20)
plt.title("Total Nilai Hilang", fontsize=20)
plt.show()


Ada tiga cara umum menangani nilai hilang. Contohnya dipakai pada kolom `Lot Frontage`.

1. Membuang baris yang kosong dengan `dropna()`.


In [ ]:
dataset.dropna(subset=["Lot Frontage"])


2. Membuang seluruh kolom yang memuat nilai kosong dengan `drop()`.


In [ ]:
dataset.drop(columns=["Lot Frontage"])


3. Mengganti nilai kosong dengan nilai tertentu, misalnya nol, rata-rata, atau median, memakai `fillna()`.


In [ ]:
median_frontage = dataset["Lot Frontage"].median()
median_frontage


In [ ]:
dataset["Lot Frontage"] = dataset["Lot Frontage"].fillna(median_frontage)


In [ ]:
dataset.tail()


Baris dengan indeks 2927 yang sebelumnya kosong pada `Lot Frontage` kini berisi nilai median.


## Latihan 4

Ganti nilai kosong pada kolom `Mas Vnr Area` dengan rata-rata kolom tersebut.


In [ ]:
mean_veneer = dataset["Mas Vnr Area"].mean()
dataset["Mas Vnr Area"] = dataset["Mas Vnr Area"].fillna(mean_veneer)
dataset["Mas Vnr Area"].isnull().sum()


## Penskalaan fitur

Penskalaan menyetarakan rentang antar fitur agar tidak ada kolom yang mendominasi hanya karena satuannya lebih besar. Ada dua pendekatan yang umum.

- **Normalisasi min-max** menggeser dan mengubah skala nilai sehingga berada pada rentang 0 sampai 1, yaitu dengan mengurangkan nilai minimum lalu membaginya dengan selisih maksimum dan minimum.
- **Standardisasi** mengurangkan rata-rata sehingga rata-rata baru menjadi nol, lalu membaginya dengan simpangan baku sehingga variansnya menjadi satu.

`scikit-learn` menyediakan `MinMaxScaler` untuk normalisasi dan `StandardScaler` untuk standardisasi.


In [ ]:
# Normalisasi seluruh kolom numerik
normalized_data = MinMaxScaler().fit_transform(numeric_data)
normalized_data


Hasilnya berupa `ndarray`. Standardisasi dilakukan dengan cara yang sama.


In [ ]:
standardized_data = StandardScaler().fit_transform(numeric_data)
standardized_data


## Latihan 5

Standardisasi kolom `SalePrice` saja memakai `StandardScaler()` dan `fit_transform()`.


In [ ]:
scaled_price = StandardScaler().fit_transform(
    dataset["SalePrice"].to_numpy().reshape(-1, 1)
)
scaled_price


## Menangani outlier

Outlier adalah pengamatan yang jaraknya jauh dari pengamatan lain. Penyebabnya bisa kesalahan pencatatan atau variasi data yang memang besar. Dampaknya pada model bisa signifikan, tetapi outlier kadang juga menyimpan informasi penting. Pendeteksiannya bisa dilakukan pada satu variabel (univariat) atau beberapa variabel sekaligus (multivariat), dan cara paling sederhana adalah memeriksa box plot atau scatter plot.

### Analisis univariat

Box plot menampilkan kuartil data. Garis yang memanjang dari kotak menunjukkan variabilitas di luar kuartil atas dan bawah, sedangkan titik di luar garis tersebut ditandai sebagai outlier. Berikut box plot untuk `Lot Area` dan `SalePrice`.


In [ ]:
sns.boxplot(x=dataset["Lot Area"])
plt.show()


In [ ]:
sns.boxplot(x=dataset["SalePrice"])
plt.show()


Kedua plot memperlihatkan titik di luar area kotak. Titik seperti ini belum tentu salah catat, jadi keputusan membuang atau mempertahankannya bergantung pada pemahaman terhadap data. Pada kasus ini titik ekstrem `Lot Area` dan `SalePrice` masih dianggap nilai yang wajar.

### Analisis bivariat

Hubungan antara `SalePrice` dan `Gr Liv Area` digambarkan dengan scatter plot.


In [ ]:
dataset.plot.scatter(x="Gr Liv Area", y="SalePrice")


Ada dua rumah dengan luas bangunan di atas 5000 kaki persegi yang tidak mengikuti pola umum. Keduanya dibuang pada latihan ini. Dua titik tinggi lainnya masih sejalan dengan tren, jadi dipertahankan.

### Menghapus outlier

Kedua baris dicari dengan mengurutkan `Gr Liv Area` dari nilai terbesar.


In [ ]:
largest_houses = dataset.sort_values(by="Gr Liv Area", ascending=False).head(2)
largest_houses


In [ ]:
outliers_removed = dataset.drop(index=largest_houses.index)
outliers_removed.shape


In [ ]:
outliers_removed.plot.scatter(x="Gr Liv Area", y="SalePrice")


Dua titik terjauh pada `Gr Liv Area` sudah tidak muncul lagi.


## Latihan 6

Periksa apakah `Lot Area` memiliki outlier. Kamu bisa memakai box plot, scatter plot antara `SalePrice` dan `Lot Area`, atau analisis Z-score. Buang outlier bila ditemukan.


In [ ]:
sns.boxplot(x=dataset["Lot Area"])
plt.show()

dataset.plot.scatter(x="Lot Area", y="SalePrice")

dataset["Lot_Area_Z"] = stats.zscore(dataset["Lot Area"])
dataset[["Lot Area", "Lot_Area_Z"]].describe().round(3)


In [ ]:
widest_lot = dataset.sort_values(by="Lot Area", ascending=False).head(1)
lot_area_removed = dataset.drop(index=widest_lot.index)
lot_area_removed.shape


Terdapat satu outlier: titik terjauh pada `Lot Area` berjarak sangat jauh dari kelompok lainnya. Nilai Z-score-nya juga melewati ambang 3, jadi baris tersebut dibuang.


## Analisis Z-score

Z-score menyatakan berapa simpangan baku jarak sebuah nilai dari rata-rata. Titik yang nilainya terlalu jauh dari nol dianggap outlier, dengan ambang yang lazim dipakai adalah 3 atau -3. Artinya, nilai di atas 3 atau di bawah -3 simpangan baku ditandai sebagai outlier.

Berikut deteksi outlier pada kolom `Low Qual Fin SF` memakai `stats.zscore` dari `scipy`.


In [ ]:
dataset["LQFSF_Z"] = stats.zscore(dataset["Low Qual Fin SF"])
dataset[["Low Qual Fin SF", "LQFSF_Z"]].describe().round(3)


Rata-rata hasil transformasi mendekati 0.000 dan simpangan bakunya 1.000, sesuai skala Z. Nilai maksimum sekitar 22.882 jauh di atas ambang +3, sehingga kolom ini memang memuat outlier.
